# Propositional logic_Artificial Intelligence (CS221)_**Build true/false statements with AND, OR, NOT. Then ask what must follow.**Some knowledge is best written as crisp true/false rules, not probabilities.     Propositional logic uses symbols that are true or false, joined by connectives.---This notebook builds a tiny propositional-logic reasoner from scratch, one piece at a time. We enumerate every possible world, keep only the ones where our rules hold, and check what *must* be true in all of them. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PythonWe'll check whether a small **knowledge base** (a set of rules) *entails* a query — meaning the query is forced to be true in every world where the rules hold. We build it in three steps: (1) write the rules and the query as functions, (2) enumerate every possible world, (3) keep only the worlds satisfying the rules and see if the query survives in all of them.

### Step 1 — Encode the knowledge base and the queryOur symbols are `P`, `Q`, `R` — each either `True` or `False`. The knowledge base is the conjunction of three facts: the implication `P → Q`, the implication `Q → R`, and the bare fact `P`. We write an implication `A → B` using its logical equivalent `(not A) or B`. The query we want to test is simply `R`.

In [ ]:
import itertools

symbols = ["P", "Q", "R"]

# Knowledge base: (P -> Q) and (Q -> R) and P.
def kb(m):
    p_implies_q = (not m["P"]) or m["Q"]
    q_implies_r = (not m["Q"]) or m["R"]
    return p_implies_q and q_implies_r and m["P"]

# Query we want to test for entailment.
def query(m):
    return m["R"]

### Step 2 — Enumerate every possible worldA **model** (or *world*) is one assignment of `True`/`False` to all three symbols. With 3 symbols there are `2³ = 8` possible worlds. `itertools.product` generates every combination, and we zip each one with the symbol names to get a dictionary like `{'P': True, 'Q': False, 'R': True}`.

In [ ]:
all_worlds = []
for vals in itertools.product([False, True], repeat=len(symbols)):
    m = dict(zip(symbols, vals))
    all_worlds.append(m)

print("total possible worlds:", len(all_worlds))

### Step 3 — Keep KB-satisfying worlds and check entailmentEntailment means: *in every world where the KB is true, the query is also true.* So we walk through the worlds, keep only those where `kb(m)` holds, and watch the query. If we ever find a KB-satisfying world where the query is false, entailment fails. If the query holds in all of them, the KB **entails** the query.

In [ ]:
models = 0
entails = True

for m in all_worlds:
    if kb(m):                            # only models where the KB is true
        models += 1
        print("KB true at", m, "-> query R =", query(m))
        if not query(m):
            entails = False

print("models satisfying KB:", models)
print("KB entails R?", entails)

## Visualize the data & results_Given the rules 'if it is raining the streets are wet' and 'if the streets are wet driving is slow', plus 'it is raining', does the knowledge base entail 'driving is slow'?_

### Step 1 — Restate the same logic with real-world namesThis is the identical reasoning as above, just with friendlier symbol names: `Raining`, `StreetsWet`, `DrivingSlow`. The two implications and the single fact `Raining` make up the knowledge base. We collect every world that satisfies it.

In [ ]:
import itertools

symbols = ["Raining", "StreetsWet", "DrivingSlow"]

def kb(m):
    r1 = (not m["Raining"]) or m["StreetsWet"]       # raining -> streets wet
    r2 = (not m["StreetsWet"]) or m["DrivingSlow"]   # streets wet -> driving slow
    return r1 and r2 and m["Raining"]

# Build the list of worlds where the KB holds (the "satisfying models").
sat = []
for v in itertools.product([False, True], repeat=3):
    m = dict(zip(symbols, v))
    if kb(m):
        sat.append(m)

print("satisfying models:", len(sat))

### Step 2 — Find which facts hold in *every* satisfying modelA fact is **entailed** only if it is true in all satisfying models. For each symbol we test whether it is `True` across every world in `sat`; `all(...)` returns `True` only when there are no counterexamples. The result tells us which facts the KB forces to be true.

In [ ]:
truth = []
for s in symbols:
    holds_everywhere = all(m[s] for m in sat)
    truth.append(int(holds_everywhere))

for s, t in zip(symbols, truth):
    print(s, "->", "true" if t else "false (not entailed)")

### Step 3 — Plot the entailed factsA bar of height 1 means the fact is true in every KB-satisfying world (entailed); a bar of height 0 means it is not. If the chain of reasoning works, `DrivingSlow` should come out entailed — the rules force it.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(symbols, truth, color=["#7ee787", "#7ee787", "#ffb454"])

for i, s in enumerate(symbols):
    label = "true" if truth[i] else "false"
    ax.text(i, truth[i], label, ha="center", va="bottom")

ax.set_ylim(0, 1.3)
ax.set_title("facts true in every KB-satisfying model")
plt.show()